# Whac-A-Mole (pygame)
Run the three cells in order. A pygame window opens; close it or press Esc to stop.

Put `mole.png` (and optionally `mole_hit.png`) in the same folder as this notebook. Without them, placeholder moles are drawn.

In [ ]:
import os, random
import pygame   # pip install pygame

WIDTH, HEIGHT, FPS = 800, 640, 60
GAME_SECONDS = 30
COLS, ROWS = 3, 3
MOLE_MAX_SIZE = (120, 110)
IMG_DIR = os.getcwd()                       # sprites live next to this notebook
MOLE_IMG, MOLE_HIT_IMG = "mole.png", "mole_hit.png"   # hit image is optional
RISE_TIME, HIDE_TIME, HIT_PAUSE, HIT_SINK = 0.14, 0.14, 0.10, 0.30

GRASS, GRASS_DARK = (96, 170, 80), (78, 148, 66)
DIRT, DIRT_LIGHT = (92, 60, 36), (130, 90, 56)
WHITE, BLACK, GOLD = (255, 255, 255), (20, 20, 20), (255, 214, 64)
HIDDEN, RISING, UP, HIDING, HIT = range(5)  # mole states

In [ ]:
# ---------------- sprites ----------------
def fit_image(img, max_size):
    w, h = img.get_size()
    s = min(max_size[0] / w, max_size[1] / h)
    return pygame.transform.smoothscale(img, (max(1, int(w * s)), max(1, int(h * s))))

def placeholder_mole(hit=False):
    w, h = MOLE_MAX_SIZE
    s = pygame.Surface((w, h), pygame.SRCALPHA)
    pygame.draw.ellipse(s, (120, 80, 50), (8, 10, w - 16, h * 2))
    pygame.draw.ellipse(s, (160, 120, 90), (w // 2 - 28, h - 55, 56, 40))
    pygame.draw.ellipse(s, (30, 20, 20), (w // 2 - 8, h - 62, 16, 11))
    for ex in (w // 2 - 24, w // 2 + 24):
        if hit:
            pygame.draw.line(s, BLACK, (ex - 6, 30), (ex + 6, 42), 3)
            pygame.draw.line(s, BLACK, (ex - 6, 42), (ex + 6, 30), 3)
        else:
            pygame.draw.circle(s, WHITE, (ex, 38), 8)
            pygame.draw.circle(s, BLACK, (ex, 38), 4)
    return s

def try_load(name):
    path = os.path.join(IMG_DIR, name)
    if os.path.exists(path):
        return fit_image(pygame.image.load(path).convert_alpha(), MOLE_MAX_SIZE)

def load_mole_images():
    normal = try_load(MOLE_IMG)
    if normal is None:
        print(f"[info] {MOLE_IMG} not found in {IMG_DIR}; using placeholder sprites.")
        return placeholder_mole(), placeholder_mole(hit=True)
    return normal, try_load(MOLE_HIT_IMG) or normal

# ---------------- moles (plain dicts) ----------------
def make_mole(base, image, hit_image):
    return {"base": base, "image": image, "normal": image, "hit": hit_image,
            "state": HIDDEN, "timer": 0.0, "up_time": 1.0, "progress": 0.0,
            "rect": pygame.Rect(0, 0, 0, 0), "clip": pygame.Rect(0, 0, 0, 0)}

def mole_hittable(m):
    return m["state"] in (RISING, UP) and m["progress"] > 0.4

def pop_up(m, up_time):
    m.update(state=RISING, timer=0.0, up_time=up_time, image=m["normal"])

def whack(m):
    m.update(state=HIT, timer=0.0, image=m["hit"])

def update_mole(m, dt):
    m["timer"] += dt
    state, t = m["state"], m["timer"]
    if state == RISING:
        m["progress"] = min(1.0, t / RISE_TIME)
        if m["progress"] >= 1.0:
            m["state"], m["timer"] = UP, 0.0
    elif state == UP:
        if t >= m["up_time"]:
            m["state"], m["timer"] = HIDING, 0.0
    elif state == HIDING:
        m["progress"] = max(0.0, 1.0 - t / HIDE_TIME)
        if m["progress"] <= 0.0:
            m["state"] = HIDDEN
    elif state == HIT:
        m["progress"] = max(0.0, 1.0 - max(0.0, t - HIT_PAUSE) / HIT_SINK)
        if t >= HIT_PAUSE + HIT_SINK:
            m["state"], m["progress"] = HIDDEN, 0.0
    w, h = m["image"].get_size()
    bx, by = m["base"]
    m["clip"] = pygame.Rect(bx - w // 2 - 4, by - h - 10, w + 8, h + 16)
    top = by + 6 - int(m["progress"] * h)
    m["rect"] = pygame.Rect(bx - w // 2, top, w, h).clip(m["clip"])

def draw_mole(screen, m):
    if m["progress"] <= 0:
        return
    w, h = m["image"].get_size()
    top = m["base"][1] + 6 - int(m["progress"] * h)
    prev = screen.get_clip()
    screen.set_clip(m["clip"])              # sprite is cut off at the ground line
    screen.blit(m["image"], (m["base"][0] - w // 2, top))
    screen.set_clip(prev)

def make_moles():
    normal, hit = load_mole_images()
    mx, top, bottom = 160, 190, 80
    return [make_mole((mx + c * (WIDTH - 2 * mx) // (COLS - 1),
                       top + r * (HEIGHT - top - bottom) // (ROWS - 1)), normal, hit)
            for r in range(ROWS) for c in range(COLS)]

# ---------------- game state ----------------
def new_game(moles, high_score=0):
    for m in moles:
        m.update(state=HIDDEN, progress=0.0)
    return {"mode": "menu", "score": 0, "hits": 0, "clicks": 0,
            "time_left": float(GAME_SECONDS), "spawn_timer": 0.5,
            "high_score": high_score, "swing": 0.0, "popups": []}

def handle_click(g, moles, pos):
    g["swing"] = 0.12
    if g["mode"] == "menu":
        g.update(new_game(moles, g["high_score"]), mode="playing")
        return
    if g["mode"] == "over":
        g["mode"] = "menu"
        return
    g["clicks"] += 1
    for m in sorted(moles, key=lambda m: m["base"][1], reverse=True):  # front first
        if mole_hittable(m) and m["rect"].collidepoint(pos):
            whack(m)
            g["score"] += 10
            g["hits"] += 1
            g["popups"].append([pos[0], pos[1] - 20, 0.7, "+10"])
            break

def update_game(g, moles, dt):
    g["swing"] = max(0.0, g["swing"] - dt)
    for p in g["popups"]:
        p[1] -= 60 * dt
        p[2] -= dt
    g["popups"] = [p for p in g["popups"] if p[2] > 0]
    if g["mode"] == "playing":
        g["time_left"] -= dt
        g["spawn_timer"] -= dt
        d = 1.0 - g["time_left"] / GAME_SECONDS      # difficulty 0 -> 1
        if g["spawn_timer"] <= 0:
            free = [m for m in moles if m["state"] == HIDDEN]
            if free:
                pop_up(random.choice(free), random.uniform(0.7, 1.2) - 0.35 * d)
            g["spawn_timer"] = random.uniform(0.45, 0.9) - 0.3 * d
        if g["time_left"] <= 0:
            g["time_left"] = 0
            g["high_score"] = max(g["high_score"], g["score"])
            g["mode"] = "over"
    for m in moles:
        update_mole(m, dt)

# ---------------- drawing ----------------
def draw_text(screen, text, font, color, center):
    s = font.render(text, True, BLACK)
    screen.blit(s, s.get_rect(center=(center[0] + 2, center[1] + 2)))
    t = font.render(text, True, color)
    screen.blit(t, t.get_rect(center=center))

def draw_hole(screen, m, front=False):
    x, y = m["base"]
    if front:   # front lip, drawn over the mole so it looks inside the hole
        pygame.draw.arc(screen, DIRT_LIGHT, (x - 78, y - 22, 156, 56), 3.4, 6.03, 7)
        return
    pygame.draw.ellipse(screen, DIRT_LIGHT, (x - 78, y - 22, 156, 56))
    pygame.draw.ellipse(screen, DIRT, (x - 68, y - 18, 136, 44))
    pygame.draw.ellipse(screen, BLACK, (x - 58, y - 14, 116, 34))

def draw_hud(screen, g, font):
    draw_text(screen, f"Score: {g['score']}", font, WHITE, (110, 30))
    draw_text(screen, f"Best: {g['high_score']}", font, GOLD, (WIDTH // 2, 30))
    color = (255, 120, 120) if g["time_left"] < 6 else WHITE
    draw_text(screen, f"Time: {int(g['time_left'] + 0.99)}", font, color, (WIDTH - 90, 30))

def draw_hammer(screen, g):
    mx, my = pygame.mouse.get_pos()
    surf = pygame.Surface((90, 90), pygame.SRCALPHA)
    pygame.draw.rect(surf, (140, 100, 60), (40, 30, 10, 58), border_radius=3)
    pygame.draw.rect(surf, (120, 120, 130), (14, 8, 62, 30), border_radius=6)
    pygame.draw.rect(surf, (80, 80, 90), (14, 8, 62, 30), 3, border_radius=6)
    rot = pygame.transform.rotate(surf, 45 if g["swing"] > 0 else -20)
    screen.blit(rot, rot.get_rect(center=(mx + 10, my - 5)))

def draw_game(screen, g, moles, fonts):
    font, big, small = fonts
    screen.fill(GRASS)
    for i in range(0, HEIGHT, 40):
        pygame.draw.rect(screen, GRASS_DARK, (0, i, WIDTH, 20))
    for m in moles:
        draw_hole(screen, m)
        draw_mole(screen, m)
        draw_hole(screen, m, front=True)
    for x, y, _, txt in g["popups"]:
        draw_text(screen, txt, font, GOLD, (x, int(y)))
    cx, cy = WIDTH // 2, HEIGHT // 2
    if g["mode"] == "playing":
        draw_hud(screen, g, font)
    elif g["mode"] == "menu":
        draw_text(screen, "WHAC-A-MOLE", big, WHITE, (cx, 90))
        draw_text(screen, "Click to start", font, WHITE, (cx, 145))
    else:
        draw_hud(screen, g, font)
        panel = pygame.Surface((460, 260), pygame.SRCALPHA)
        panel.fill((0, 0, 0, 170))
        screen.blit(panel, panel.get_rect(center=(cx, cy)))
        acc = int(100 * g["hits"] / g["clicks"]) if g["clicks"] else 0
        draw_text(screen, "TIME'S UP!", big, GOLD, (cx, cy - 80))
        draw_text(screen, f"Score: {g['score']}", font, WHITE, (cx, cy - 15))
        draw_text(screen, f"Accuracy: {acc}%", small, WHITE, (cx, cy + 25))
        draw_text(screen, "Click to continue", small, WHITE, (cx, cy + 75))
    draw_hammer(screen, g)
    pygame.display.flip()

In [ ]:
def main(max_frames=None):
    pygame.init()
    try:
        screen = pygame.display.set_mode((WIDTH, HEIGHT))
        pygame.display.set_caption("Whac-A-Mole  (Esc or close the window to quit)")
        clock = pygame.time.Clock()
        fonts = (pygame.font.SysFont("arial", 28, bold=True),
                 pygame.font.SysFont("arial", 64, bold=True),
                 pygame.font.SysFont("arial", 22))
        pygame.mouse.set_visible(False)
        moles = make_moles()            # after set_mode (convert_alpha needs it)
        g = new_game(moles)
        frames, running = 0, True
        while running:
            dt = clock.tick(FPS) / 1000.0
            for e in pygame.event.get():
                if e.type == pygame.QUIT or (e.type == pygame.KEYDOWN and e.key == pygame.K_ESCAPE):
                    running = False
                elif e.type == pygame.MOUSEBUTTONDOWN and e.button == 1:
                    handle_click(g, moles, e.pos)
                elif e.type == pygame.KEYDOWN and e.key == pygame.K_SPACE and g["mode"] != "playing":
                    handle_click(g, moles, pygame.mouse.get_pos())
            update_game(g, moles, dt)
            draw_game(screen, g, moles, fonts)
            frames += 1
            if max_frames and frames >= max_frames:
                break
    finally:
        pygame.quit()                   # always close the window, even on errors

main()